In [22]:
import numpy as np
import plotly.graph_objects as go

from scipy.optimize import minimize
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [23]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=120,
    n_features=3,
    n_informative=3,
    n_redundant=0,
    n_classes=2,
    n_clusters_per_class=1,
    class_sep=0.8,
    flip_y=0.08,
    random_state=42
)

y = np.where(y == 0, -1, 1)

In [24]:
class SoftMarginSVM:

    def __init__(self, C=1.0):

        self.C = C

        self.w = None
        self.b = None
        self.xi = None

        self.support_vectors_ = None
        self.support_indices_ = None


    def fit(self, X, y):

        n_samples, n_features = X.shape

        theta_initial = np.zeros(n_features + 1 + n_samples)


        def objective(theta):
            w = theta[:n_features]
            xi = theta[n_features + 1:]
            return (0.5 * np.dot(w, w)+ self.C * np.sum(xi))

        def margin_constraint(theta):
            w = theta[:n_features]
            b = theta[n_features]
            xi = theta[n_features + 1:]
            decision = X @ w + b
            return y * decision + xi - 1

        constraints = [
            {"type": "ineq","fun": margin_constraint}
        ]
        bounds = []
        
        for _ in range(n_features):
            bounds.append((None, None))
            
        bounds.append((None, None))

        for _ in range(n_samples):
            bounds.append((0, None))

        result = minimize(
            objective,
            theta_initial,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={
                "ftol": 1e-9,
                "maxiter": 3000
            }
        )


        if not result.success:
            raise RuntimeError(result.message)

        self.w = result.x[:n_features]
        self.b = result.x[n_features]
        self.xi = result.x[n_features + 1:]

        margins = y * (X @ self.w + self.b)
        self.support_indices_ = np.where(margins <= 1 + 1e-4)[0]
        self.support_vectors_ = X[self.support_indices_]

        return self


    def decision_function(self, X):
        return X @ self.w + self.b


    def predict(self, X):

        scores = self.decision_function(X)
        return np.where(scores >= 0, 1, -1)

In [25]:
C = 1.0

scratch_svm = SoftMarginSVM(C=C)

scratch_svm.fit(X, y)

print("Scratch Soft-Margin SVM")

print("w =", scratch_svm.w)
print("b =", scratch_svm.b)

print("C =", C)

print(
    "Number of margin/violating points:",
    len(scratch_svm.support_vectors_)
)

Scratch Soft-Margin SVM
w = [1.58582118 1.71561973 0.0114454 ]
b = 1.4131428069891745
C = 1.0
Number of margin/violating points: 40


In [26]:
sklearn_svm = SVC(
    kernel="linear",
    C=C
)

sklearn_svm.fit(X, y)


print("\nSklearn SVM")

print("w =", sklearn_svm.coef_[0])

print("b =", sklearn_svm.intercept_[0])

print(
    "Number of support vectors:",
    len(sklearn_svm.support_)
)


Sklearn SVM
w = [1.58669897 1.71617255 0.01187398]
b = 1.4135677174185237
Number of support vectors: 40


In [27]:
scratch_pred = scratch_svm.predict(X)

sklearn_pred = sklearn_svm.predict(X)


print(
    "\nScratch accuracy:",
    accuracy_score(y, scratch_pred)
)

print(
    "Sklearn accuracy:",
    accuracy_score(y, sklearn_pred)
)


Scratch accuracy: 0.9166666666666666
Sklearn accuracy: 0.9166666666666666


In [28]:
w = scratch_svm.w
b = scratch_svm.b

x_range = np.linspace(
    X[:, 0].min(),
    X[:, 0].max(),
    30
)

y_range = np.linspace(
    X[:, 1].min(),
    X[:, 1].max(),
    30
)

xx, yy = np.meshgrid(
    x_range,
    y_range
)


def plane_z(offset):

    return (
        offset
        - b
        - w[0] * xx
        - w[1] * yy
    ) / w[2]


z_decision = plane_z(0)
z_positive = plane_z(1)
z_negative = plane_z(-1)
fig = go.Figure()

fig.add_trace(
    go.Surface(
        x=xx,
        y=yy,
        z=z_decision,
        opacity=0.35,
        showscale=False,
        name="Decision Plane"
    )
)

fig.add_trace(
    go.Surface(
        x=xx,
        y=yy,
        z=z_positive,
        opacity=0.25,
        showscale=False,
        name="+1 Margin"
    )
)
fig.add_trace(
    go.Surface(
        x=xx,
        y=yy,
        z=z_negative,
        opacity=0.25,
        showscale=False,
        name="-1 Margin"
    )
)

mask_neg = y == -1

fig.add_trace(
    go.Scatter3d(
        x=X[mask_neg, 0],
        y=X[mask_neg, 1],
        z=X[mask_neg, 2],
        mode="markers",
        name="Class -1",
        marker=dict(
            size=5
        )
    )
)
mask_pos = y == 1

fig.add_trace(
    go.Scatter3d(
        x=X[mask_pos, 0],
        y=X[mask_pos, 1],
        z=X[mask_pos, 2],
        mode="markers",
        name="Class +1",
        marker=dict(
            size=5
        )
    )
)

sv = scratch_svm.support_indices_

fig.add_trace(
    go.Scatter3d(
        x=X[sv, 0],
        y=X[sv, 1],
        z=X[sv, 2],
        mode="markers",
        name="Support / Margin Points",
        marker=dict(
            size=10,
            symbol="diamond",
            line=dict(width=2)
        )
    )
)

fig.update_layout(
    title="Soft-Margin SVM — Decision & Margin Planes",

    scene=dict(
        xaxis_title="X₁",
        yaxis_title="X₂",
        zaxis_title="X₃"
    ),

    width=900,
    height=700
)

fig.show()